In [ ]:
#| default_exp tree

# tree
> tree data structure helpers

In [ ]:
#| export
from pathlib import Path
from typing import Mapping, Sequence, Hashable, Callable, Literal, Any
from operator import attrgetter

from fastcore.foundation import L, AttrDict, patch, noop
from fasthtml.components import Details, Summary, Ul, Li, show

from pote.basic import is_listy, val_atpath

In [ ]:
import json
from copy import deepcopy
from IPython.display import HTML

import fastcore.all as FC
from fastcore.test import *
from fastcore.xml import FT
from fasthtml.xtend import Style
from fasthtml.components import Div, Span

from pote.basic import val_at, vals_at, vals_atpath, is_listy, is_listy_type, flatten

## tree data structure

In [ ]:
#| export
# Pre 3.12+
# from typing import Any, TypeAlias
# Tree: TypeAlias = dict[str, list[Any | "Tree"]]

type Tree = Mapping[Hashable, Sequence[Any | Tree]]

In [ ]:
tr: Tree = {
    'Apollo astronauts': [
        'Neil Armstrong',
        'Alan Bean',
        {
            'other': [
                'Bruce Wayne',
                'Clark Kent',
                'Peter Parker'
            ]
        },
        'Edgar Mitchell', 
        'Alan Shepard'
    ],
    'Apollo 11': [
        'Neil Armstrong', 
        'Buzz Aldrin', 
        'Edgar Mitchell', 
        {
            'a': 1,
            'b': 2
        },
        'Alan Shepard'
    ]
}


In [ ]:
test_eq(val_at(tr, 'Apollo astronauts.0'), 'Neil Armstrong')
test_eq(val_at(tr, 'Apollo astronauts.2.other.1'), 'Clark Kent')
test_eq(val_at(tr, 'Apollo 11.3.a'), 1)
test_eq(val_at(tr, 'Apollo 11.3.b'), 2)

The `tree` data structure:

1. **Forest, not tree**: The root is a `dict` with multiple named keys, each mapping to a list of children — making it a *forest* of named trees rather than a single-rooted tree.

2. **Non-leaf nodes are dicts, leaves are anything else**: A subtree is represented as a `dict` (with a name → children-list mapping), and any non-dict value is a leaf. No custom classes needed.

3. **JSON-compatible by design**: Since the structure only uses dicts, lists, strings, and numbers in its basic form, it round-trips cleanly with `json.dumps`/`json.loads` out of the box.

Alternatives in the python ecosystem to handle trees (and graphs)? from lightweight to full-featured:

**Built-in / stdlib**
- `dict` + `list` — what you're doing now; zero deps, JSON-compatible
- `collections.namedtuple` / `dataclasses` — slightly more structured, still no tree logic

**Lightweight tree libs**
- [`anytree`](https://anytree.readthedocs.io/) — simple, pure Python, good for named nodes, has walkers/renderers/exporters (including JSON/dict). Probably the closest to your use case.
- [`treelib`](https://treelib.readthedocs.io/) — node-based tree with IDs, good for display/traversal, no graph support
- [`bigtree`](https://bigtree.readthedocs.io/) — newer, pandas-friendly, dict/list/JSON import-export

**General-purpose graph libs (trees are a subset)**
- [`networkx`](https://networkx.org/) — the standard; full graph theory, algorithms, traversals. Trees are just DAGs. Heavy but comprehensive.
- [`igraph`](https://python.igraph.org/) — faster than networkx for large graphs, C backend

**Functional / recursive style**
- [`toolz`] — no tree type, but functional primitives that compose well for recursive traversal
- [`glom`](https://glom.readthedocs.io/) — path-based nested data access, similar to your `val_at`, no tree structure per se

**Specialized**
- [`lxml`](https://lxml.de/) / `xml.etree` — if your tree maps to XML
- [`rich`](https://rich.readthedocs.io/) — has a `Tree` class but purely for rendering, not data
- [`rustworkx`](https://www.rustworkx.org/) — Rust-backed graph lib from Qiskit, very fast

---

Given the constraints (no custom class, JSON-compatible, named nodes), `anytree` is the most relevant ready-made solution. But the dict-of-lists approach is simpler and more transparent for our specific shape.


## walker

A simple walker (depth-first). Recursive first, then stack-based. Paths will be consistent with `val_at`.

- ('Apollo astronauts.0', 'Neil Armstrong')
- ('Apollo astronauts.1', 'Alan Bean')
- ('Apollo astronauts.2.other.1', 'Bruce Wayne')
...

In [ ]:
def walk(m: Tree, p: str=''):
    "Depth-first walk of a forest, yielding (path, leaf) pairs"
    for k, v in m.items():
        cp = f"{p}.{k}" if p else k
        if is_listy(v):
            for i, item in enumerate(v):
                if isinstance(item, Mapping): yield from walk(item, f"{cp}.{i}")
                else: yield f"{cp}.{i}", item
        else: yield cp, v

In [ ]:
list(walk(tr))

[('Apollo astronauts.0', 'Neil Armstrong'),
 ('Apollo astronauts.1', 'Alan Bean'),
 ('Apollo astronauts.2.other.0', 'Bruce Wayne'),
 ('Apollo astronauts.2.other.1', 'Clark Kent'),
 ('Apollo astronauts.2.other.2', 'Peter Parker'),
 ('Apollo astronauts.3', 'Edgar Mitchell'),
 ('Apollo astronauts.4', 'Alan Shepard'),
 ('Apollo 11.0', 'Neil Armstrong'),
 ('Apollo 11.1', 'Buzz Aldrin'),
 ('Apollo 11.2', 'Edgar Mitchell'),
 ('Apollo 11.3.a', 1),
 ('Apollo 11.3.b', 2),
 ('Apollo 11.4', 'Alan Shepard')]

The stack-based version:

In [ ]:
def walk_s(m: Tree, p: str=''):
    "Depth-first walk of a forest, yielding (path, leaf) pairs"
    stack = [(f"{p}.{k}" if p else k, v) for k, v in reversed(list(m.items()))]
    while stack:
        cp, v = stack.pop()
        if isinstance(v, Mapping):
            for k, vv in reversed(list(v.items())): stack.append((f"{cp}.{k}", vv))
        elif is_listy(v):
            for i, item in reversed(list(enumerate(v))): stack.append((f"{cp}.{i}", item))
        else: yield cp, v

In [ ]:
w = list(walk_s(tr))
test_eq(w, list(walk(tr)))

In [ ]:
def walk(m: Tree, p: str='', order: Literal['depth', 'breadth']='depth'):
    "Walk a forest yielding (path, leaf) pairs"
    if order == 'depth':
        for k, vs in m.items():
            cp = f"{p}.{k}" if p else k
            for i, v in enumerate(vs if is_listy(vs) else [vs]):
                if isinstance(v, Mapping): yield from walk(v, f"{cp}.{i}", order)
                else: yield f"{cp}.{i}", v
    else:
        queue = [(f"{p}.{k}" if p else k, vs) for k, vs in m.items()]
        while queue:
            cp, vs = queue.pop(0)
            for i, v in enumerate(vs if is_listy(vs) else [vs]):
                if isinstance(v, Mapping): queue.extend((f"{cp}.{i}.{k}", vv) for k, vv in v.items())
                else: yield f"{cp}.{i}", v

In [ ]:
list(walk(tr, order='breadth'))

[('Apollo astronauts.0', 'Neil Armstrong'),
 ('Apollo astronauts.1', 'Alan Bean'),
 ('Apollo astronauts.3', 'Edgar Mitchell'),
 ('Apollo astronauts.4', 'Alan Shepard'),
 ('Apollo 11.0', 'Neil Armstrong'),
 ('Apollo 11.1', 'Buzz Aldrin'),
 ('Apollo 11.2', 'Edgar Mitchell'),
 ('Apollo 11.4', 'Alan Shepard'),
 ('Apollo astronauts.2.other.0', 'Bruce Wayne'),
 ('Apollo astronauts.2.other.1', 'Clark Kent'),
 ('Apollo astronauts.2.other.2', 'Peter Parker'),
 ('Apollo 11.3.a.0', 1),
 ('Apollo 11.3.b.0', 2)]

## search/filter

A few natural directions here:

- **search by value** — find all `(path, leaf)` pairs where the leaf matches a predicate
- **search by path** — find all pairs where the path matches a pattern (e.g. glob or regex)
- **prune** — return a new tree with only the branches that match a predicate
- **filter leaves** — like prune but simpler, just dropping non-matching leaves

In [ ]:
#| export
def search(m: Tree, pred: Callable):
    "Find all (path, leaf) pairs where leaf satisfies pred"
    return [(p, v) for p, v in walk(m) if pred(v)]

In [ ]:
test_eq(search(tr, lambda v: 'Armstrong' in str(v)),
    [('Apollo astronauts.0', 'Neil Armstrong'), ('Apollo 11.0', 'Neil Armstrong')])

test_eq(search(tr, lambda v: 'Turner' in str(v)), [])

In [ ]:
#| export
def search_path(m: Tree, pred: Callable):
    "Find all (path, leaf) pairs where path satisfies pred"
    return [(p, v) for p, v in walk(m) if pred(p)]

In [ ]:
test_eq(search_path(tr, lambda p: 'Apollo 11' in p),
    [('Apollo 11.0', 'Neil Armstrong'),
    ('Apollo 11.1', 'Buzz Aldrin'),
    ('Apollo 11.2', 'Edgar Mitchell'),
    ('Apollo 11.3.a.0', 1),
    ('Apollo 11.3.b.0', 2),
    ('Apollo 11.4', 'Alan Shepard')])

Two related operations:

**filter leaves** — keep only leaves matching a predicate, empty branches remain

**prune** — same, but also removes branches that end up empty


In [ ]:
def filter_leaves(m: Tree, pred: Callable) -> Tree:
    "Keep only leaves satisfying pred, empty branches remain"
    return {k: [filter_leaves(v, pred) if isinstance(v, Mapping) else v 
                for v in vs if isinstance(v, Mapping) or pred(v)]
            if is_listy(vs) else vs
            for k, vs in m.items()}

In [ ]:
test_eq(filter_leaves(tr, lambda v: 'Armstrong' in str(v)),
    {'Apollo astronauts': ['Neil Armstrong', {'other': []}],
    'Apollo 11': ['Neil Armstrong', {'a': 1, 'b': 2}]})

In [ ]:
def prune(m: Tree, pred: Callable) -> Tree:
    "Keep only leaves satisfying pred, removing empty branches"
    res = {}
    for k, vs in m.items():
        if not is_listy(vs): res[k] = vs; continue
        children = []
        for v in vs:
            if isinstance(v, Mapping):
                sub = prune(v, pred)
                if sub: children.append(sub)
            elif pred(v): children.append(v)
        if children: res[k] = children
    return res

In [ ]:
test_eq(prune(tr, lambda v: 'Armstrong' in str(v)),
    {'Apollo astronauts': ['Neil Armstrong'],
    'Apollo 11': ['Neil Armstrong', {'a': 1, 'b': 2}]})

## what's a tree

A QoL helper `is_tree(...)`.

In [ ]:
def is_tree(o) -> bool:
    "Return True if o is a Mapping with all list values"
    return isinstance(o, Mapping) and all(is_listy(v) for v in o.values())

In [ ]:
test_is(is_tree({}), True)
test_is(is_tree({'a': 1}), False)
test_is(is_tree({'a': [], 'b':[]}), True)

Now it's clear what's a tree. `is_tree` checked for a `Mapping` with *all list values*, which felt wrong — `{'a': 1}` was rejected, yet it clearly represents a tree node with a scalar child.

The cleaner invariant:

- **`Mapping` → always a tree node**, regardless of value types
- **anything else → always a leaf**

So `isinstance(o, Mapping)` is the complete definition of a tree node. A value like `{'a': 1}` is a subtree with `1` as a child under key `'a'`, not a leaf. This also simplifies the walker and filter functions, removing the `is_listy(vs) else vs` guards since every mapping value can be treated uniformly as a (possibly scalar) child.

In [ ]:
#| export
def is_tree(o) -> bool: return isinstance(o, Mapping)

In [ ]:
def walk(m: Tree, p: str=''):
    "Depth-first walk of a forest, yielding (path, leaf) pairs"
    for k, vs in m.items():
        cp = f"{p}.{k}" if p else k
        for i, item in enumerate(vs if is_listy(vs) else [vs]):
            if is_tree(item): yield from walk(item, f"{cp}.{i}")
            else: yield f"{cp}.{i}", item

In [ ]:
list(walk(tr))

[('Apollo astronauts.0', 'Neil Armstrong'),
 ('Apollo astronauts.1', 'Alan Bean'),
 ('Apollo astronauts.2.other.0', 'Bruce Wayne'),
 ('Apollo astronauts.2.other.1', 'Clark Kent'),
 ('Apollo astronauts.2.other.2', 'Peter Parker'),
 ('Apollo astronauts.3', 'Edgar Mitchell'),
 ('Apollo astronauts.4', 'Alan Shepard'),
 ('Apollo 11.0', 'Neil Armstrong'),
 ('Apollo 11.1', 'Buzz Aldrin'),
 ('Apollo 11.2', 'Edgar Mitchell'),
 ('Apollo 11.3.a.0', 1),
 ('Apollo 11.3.b.0', 2),
 ('Apollo 11.4', 'Alan Shepard')]

In [ ]:
def walk_s(m: Tree, p: str=''):
    "Depth-first walk of a forest, yielding (path, leaf) pairs"
    stack = [(f"{p}.{k}" if p else k, v) for k, v in reversed(list(m.items()))]
    while stack:
        cp, v = stack.pop()
        if is_tree(v):
            for k, vv in reversed(list(v.items())):
                vs = vv if is_listy(vv) else [vv]
                for i, item in reversed(list(enumerate(vs))): stack.append((f"{cp}.{k}.{i}", item))
        elif is_listy(v):
            for i, item in reversed(list(enumerate(v))): stack.append((f"{cp}.{i}", item))
        else: yield cp, v

In [ ]:
w = walk_s(tr)
test_eq(w, walk(tr))

In [ ]:
#| export
def filter_leaves(m: Tree, pred: Callable) -> Tree:
    "Keep only leaves satisfying pred, empty branches remain"
    return {k: [filter_leaves(v, pred) if is_tree(v) else v
                for v in (vs if is_listy(vs) else [vs]) if is_tree(v) or pred(v)]
            for k, vs in m.items()}

In [ ]:
#| export
def prune(m: Tree, pred: Callable) -> Tree:
    "Keep only leaves satisfying pred, removing empty branches"
    res = {}
    for k, vs in m.items():
        children = []
        for v in (vs if is_listy(vs) else [vs]):
            if is_tree(v):
                sub = prune(v, pred)
                if sub: children.append(sub)
            elif pred(v): children.append(v)
        if children: res[k] = children
    return res

In [ ]:
test_eq(filter_leaves(tr, lambda v: 'Armstrong' in str(v)),
    {'Apollo astronauts': ['Neil Armstrong', {'other': []}],
    'Apollo 11': ['Neil Armstrong', {'a': [], 'b': []}]})

In [ ]:
test_eq(prune(tr, lambda v: 'Armstrong' in str(v)),
    {'Apollo astronauts': ['Neil Armstrong'], 'Apollo 11': ['Neil Armstrong']})

## transform

A general `transform` that walks the tree and applies callbacks at each node/leaf.

In [ ]:
def transform(m: Tree, on_leaf: Callable=None, on_node_pre: Callable=None, on_node: Callable=None, p: str='') -> Tree:
    "Walk forest applying callbacks, returning new tree"
    res = {}
    for k, vs in m.items():
        children, cp, wrapped = [], f"{p}.{k}" if p else k, not is_listy(vs)
        for i, v in enumerate(vs if not wrapped else [vs]):
            ip = f"{cp}.{i}"
            if is_tree(v):
                if on_node_pre: on_node_pre(ip, v)
                sv = transform(v, on_leaf, on_node_pre, on_node, ip)
                r = on_node(ip, sv) if on_node else None
                children.append(r if r is not None else sv)
            else:
                r = on_leaf(ip, v) if on_leaf else None
                children.append(r if r is not None else v)
        res[k] = children[0] if wrapped else children
    if on_node and res:
        r = on_node(p, res)
        if r is not None: res = r
    return res

In [ ]:
test_eq(transform(tr), tr)

In [ ]:
trd = transform(tr, on_leaf=lambda p, v: v.upper() if isinstance(v, str) else v)
test_eq(trd,
    {'Apollo astronauts': ['NEIL ARMSTRONG', 'ALAN BEAN',
        {'other': ['BRUCE WAYNE', 'CLARK KENT', 'PETER PARKER']},
        'EDGAR MITCHELL', 'ALAN SHEPARD'],
    'Apollo 11': ['NEIL ARMSTRONG', 'BUZZ ALDRIN', 'EDGAR MITCHELL',
        {'a': 1, 'b': 2},
        'ALAN SHEPARD']}
)

In [ ]:
trd = transform(tr, on_leaf=lambda p, v: v.upper() if isinstance(v, str) else v, on_node=lambda p, v: {k.upper(): v[k] for k in v.keys()})
test_eq(trd.keys(), ('APOLLO ASTRONAUTS', 'APOLLO 11'))

In [ ]:
transform(tr, on_leaf=lambda p, v: f"{p}={v}")

{'Apollo astronauts': ['Apollo astronauts.0=Neil Armstrong',
  'Apollo astronauts.1=Alan Bean',
  {'other': ['Apollo astronauts.2.other.0=Bruce Wayne',
    'Apollo astronauts.2.other.1=Clark Kent',
    'Apollo astronauts.2.other.2=Peter Parker']},
  'Apollo astronauts.3=Edgar Mitchell',
  'Apollo astronauts.4=Alan Shepard'],
 'Apollo 11': ['Apollo 11.0=Neil Armstrong',
  'Apollo 11.1=Buzz Aldrin',
  'Apollo 11.2=Edgar Mitchell',
  {'a': 'Apollo 11.3.a.0=1', 'b': 'Apollo 11.3.b.0=2'},
  'Apollo 11.4=Alan Shepard']}

In [ ]:
def pprint(m: Tree, indent: int=2):
    "Pretty print a forest"
    def _pre(p, v): print(' '*indent*p.count('.') + list(v.keys())[0] + ':')
    def _leaf(p, v): print(' '*indent*p.count('.') + str(v))
    for k, vs in m.items():
        print(f"{k}:")
        transform({k: vs}, on_leaf=_leaf, on_node_pre=_pre)

In [ ]:
pprint(tr)

Apollo astronauts:
  Neil Armstrong
  Alan Bean
  other:
      Bruce Wayne
      Clark Kent
      Peter Parker
  Edgar Mitchell
  Alan Shepard
Apollo 11:
  Neil Armstrong
  Buzz Aldrin
  Edgar Mitchell
  a:
      1
      2
  Alan Shepard


## generalized tree

A generalized tree where keys (until now considered "names") are hashable objects, not only strings. For instance, `pathlib.Path` (hashable in later Pythons). In that way, it can't perfectly well represent a file system.

The data structure itself already supports it since Python dicts accept any hashable key.

The main implication is the **path representation** — dot-separated strings assume string keys. With `Path` keys for example, you'd want paths like `Path('home/user/docs')` rather than `'home.user.0.docs.1'`.

So, represent paths as **tuples** of keys — more general, round-trippable, works with any hashable

`val_atpath` already uses a tuple-of-keys path — perfect fit. So paths in the generalized tree would be tuples of hashable keys (with integer indices for list positions), and we already have the traversal function to resolve them.

And `walk` would yield `(tuple_path, leaf)` pairs instead of dot-strings.

In [ ]:
#| export
def walk(m: Tree, p: tuple=(), order: Literal['depth', 'breadth']='depth'):
    "Walk a forest yielding (path, leaf) pairs with tuple paths"
    if order == 'depth':
        for k, vs in m.items():
            cp = p + (k,)
            for i, v in enumerate(vs if is_listy(vs) else [vs]):
                if is_tree(v): yield from walk(v, cp + (i,), order)
                else: yield cp + (i,), v
    else:
        queue = [(p + (k,), vs) for k, vs in m.items()]
        while queue:
            cp, vs = queue.pop(0)
            for i, v in enumerate(vs if is_listy(vs) else [vs]):
                if is_tree(v): queue.extend([(cp + (i, k), vv) for k, vv in v.items()])
                else: yield cp + (i,), v

In [ ]:
list(walk(tr, order='breadth'))

[(('Apollo astronauts', 0), 'Neil Armstrong'),
 (('Apollo astronauts', 1), 'Alan Bean'),
 (('Apollo astronauts', 3), 'Edgar Mitchell'),
 (('Apollo astronauts', 4), 'Alan Shepard'),
 (('Apollo 11', 0), 'Neil Armstrong'),
 (('Apollo 11', 1), 'Buzz Aldrin'),
 (('Apollo 11', 2), 'Edgar Mitchell'),
 (('Apollo 11', 4), 'Alan Shepard'),
 (('Apollo astronauts', 2, 'other', 0), 'Bruce Wayne'),
 (('Apollo astronauts', 2, 'other', 1), 'Clark Kent'),
 (('Apollo astronauts', 2, 'other', 2), 'Peter Parker'),
 (('Apollo 11', 3, 'a', 0), 1),
 (('Apollo 11', 3, 'b', 0), 2)]

In [ ]:
#| export
def walk_s(m: Tree, p: tuple=(), order: Literal['depth', 'breadth']='depth'):
    "Walk a forest using a stack/queue, yielding (path, leaf) pairs with tuple paths"
    init = [(p + (k,), v) for k, v in (reversed(list(m.items())) if order == 'depth' else m.items())]
    buf = list(init)
    while buf:
        cp, v = buf.pop() if order == 'depth' else buf.pop(0)
        if is_tree(v):
            items = reversed(list(v.items())) if order == 'depth' else v.items()
            for k, vv in items:
                vs = vv if is_listy(vv) else [vv]
                idxs = reversed(list(enumerate(vs))) if order == 'depth' else enumerate(vs)
                for i, item in idxs: buf.append((cp + (k, i), item))
        elif is_listy(v):
            idxs = reversed(list(enumerate(v))) if order == 'depth' else enumerate(v)
            for i, item in idxs: buf.append((cp + (i,), item))
        else: yield cp, v

In [ ]:
test_eq(walk_s(tr, order='breadth'), walk(tr, order='breadth'))

## more CRUD
Structural mutation operations.

- **`insert`** — add a leaf or subtree at a given path
- **`remove`** — delete a leaf or subtree at a given path
- **`update`** — replace a leaf or subtree at a given path
- **`move`** — relocate a node from one path to another (remove + insert)
- **`rename`** — rename a node key without moving its children

In [ ]:
tr2 = deepcopy(tr)

In [ ]:
#| export
def rename(m: Tree, path, new_key):
    "Rename a node key in-place at path; raises if not found or new_key exists"
    *parent_path, old_key = path if is_listy(path) else path.split('.')
    node = val_atpath(m, *parent_path) if parent_path else m
    if old_key not in node: raise KeyError(f"{old_key!r} not found")
    if new_key in node: raise KeyError(f"{new_key!r} already exists")
    node[new_key] = node.pop(old_key)

In [ ]:
tr2 = deepcopy(tr)

# Normal: rename root key
rename(tr2, ['Apollo astronauts'], 'Apollo moonwalkers')
test_eq(val_at(tr2, 'Apollo moonwalkers'), tr['Apollo astronauts'])

# Normal: rename nested subtree key
rename(tr2, ['Apollo moonwalkers', 2, 'other'], 'fictional')
test_eq(val_at(tr2, 'Apollo moonwalkers.2.fictional'), val_at(tr, 'Apollo astronauts.2.other'))

# Edge: rename to existing key → raises KeyError
test_fail(lambda:rename(tr2, ['Apollo 11'], 'Apollo moonwalkers'))

# Edge: rename non-existent key → raises KeyError
test_fail(lambda:rename(tr2, ['Apollo XI'], 'ghost'))

# # Edge: single-level path (root)
rename(tr2, ['Apollo 11'], 'Apollo XI')
test_eq(val_at(tr2, 'Apollo XI'), val_at(tr, 'Apollo 11'))

In [ ]:
#| export
def remove(m: Tree, path):
    "Remove node or leaf at path in-place; raises if not found"
    *parent_path, last = path if is_listy(path) else path.split('.')
    node = val_atpath(m, *parent_path) if parent_path else m
    if isinstance(node, Mapping): del node[last]
    elif is_listy(node): node.pop(int(last))
    else: raise KeyError(f"Cannot remove from {type(node)}")

In [ ]:
tr2 = deepcopy(tr)

# Normal: remove a leaf by index
remove(tr2, ['Apollo astronauts', 0])
test_eq(val_at(tr2, 'Apollo astronauts.0'), 'Alan Bean')

# Normal: remove a subtree key from the now-index-1 dict
remove(tr2, ['Apollo astronauts', 1, 'other'])
test_eq(val_at(tr2, 'Apollo astronauts.1'), {})

# Normal: remove a root key
remove(tr2, ['Apollo 11'])
test_fail(lambda: val_at(tr2, 'Apollo 11.0'))

# Edge: remove non-existent key → raises
test_fail(lambda: remove(tr2, ['ghost']))

# Edge: out of range index → raises
test_fail(lambda: remove(tr2, ['Apollo astronauts', 99]))

In [ ]:
tr2

{'Apollo astronauts': ['Alan Bean', {}, 'Edgar Mitchell', 'Alan Shepard']}

In [ ]:
#| export
def clean(m: Tree):
    "Remove empty branches in-place recursively"
    for k in list(m):
        vs = m[k]
        if is_listy(vs):
            for v in vs:
                if is_tree(v): clean(v)
            m[k] = [v for v in vs if not ((is_tree(v) or is_listy(v)) and not v)]
            if not m[k]: del m[k]
        elif is_tree(vs) and not vs: del m[k]

In [ ]:
tr2 = deepcopy(tr)
tr2['Apollo astronauts'][2] = {}
tr2['Apollo 11'].append([])
clean(tr2)
test_eq(tr2['Apollo astronauts'], ['Neil Armstrong', 'Alan Bean', 'Edgar Mitchell', 'Alan Shepard'])
test_eq(tr2['Apollo 11'], tr['Apollo 11'])

tr2 = deepcopy(tr)
tr2['empty'] = []
clean(tr2)
test_is('empty' in tr2, False)

tr2 = {'a': [{'b': []}]}
clean(tr2)
test_eq(tr2, {})

In [ ]:
#| export
def insert(m: Tree, path, value):
    "Insert value at path in-place; raises if already exists or parent not found"
    *parent_path, last = path if is_listy(path) else path.split('.')
    node = val_atpath(m, *parent_path) if parent_path else m
    if isinstance(node, Mapping):
        if last in node: raise KeyError(f"{last!r} already exists")
        node[last] = value
    elif is_listy(node):
        try: node.insert(int(last), value)
        except ValueError: node.append({last: value})
    else: raise KeyError(f"Cannot insert into {type(node)}")

In [ ]:
tr2 = deepcopy(tr)

# Normal: insert new root key
insert(tr2, ['Gemini astronauts'], ['Gus Grissom', 'John Glenn'])
test_eq(val_at(tr2, 'Gemini astronauts.0'), 'Gus Grissom')

# Normal: insert leaf at index
insert(tr2, ['Apollo 11', 0], 'Michael Collins')
test_eq(val_at(tr2, 'Apollo 11.0'), 'Michael Collins')

# Normal: insert subtree
insert(tr2, ['Apollo astronauts', 2, 'fictional2'], ['Batman', 'Superman'])
test_eq(val_at(tr2, 'Apollo astronauts.2.fictional2.0'), 'Batman')

# Edge: insert existing root key → raises
test_fail(lambda: insert(tr2, ['Apollo 11'], []))

# Edge: parent not found → raises
test_fail(lambda: insert(tr2, ['ghost', 'key'], 'value'))

In [ ]:
#| export
def update(m: Tree, path, value):
    "Update value at path in-place; raises if not found"
    *parent_path, last = path if is_listy(path) else path.split('.')
    node = val_atpath(m, *parent_path) if parent_path else m
    if isinstance(node, Mapping):
        if last not in node: raise KeyError(f"{last!r} not found")
        node[last] = value
    elif is_listy(node): node[int(last)] = value
    else: raise KeyError(f"Cannot update {type(node)}")

In [ ]:
tr2 = deepcopy(tr)

# Normal: update a leaf
update(tr2, ['Apollo astronauts', 0], 'Michael Collins')
test_eq(val_at(tr2, 'Apollo astronauts.0'), 'Michael Collins')

# Normal: replace a subtree
update(tr2, ['Apollo astronauts', 2], {'new': ['X', 'Y']})
test_eq(val_at(tr2, 'Apollo astronauts.2.new.0'), 'X')

# Normal: update a root key's value
update(tr2, ['Apollo 11'], ['Only one'])
test_eq(val_at(tr2, 'Apollo 11.0'), 'Only one')

# Edge: update non-existent key → raises
test_fail(lambda: update(tr2, ['ghost'], 'value'))

# Edge: index out of range → raises
test_fail(lambda: update(tr2, ['Apollo astronauts', 99], 'X'))

In [ ]:
#| export
def move(m: Tree, src, dst):
    "Move node at src path to dst path in-place; raises if src not found or dst exists"
    *sp, sk = src if is_listy(src) else src.split('.')
    snode = val_atpath(m, *sp) if sp else m
    if isinstance(snode, Mapping):
        if sk not in snode: raise KeyError(f"{sk!r} not found")
        value = snode.pop(sk)
    elif is_listy(snode): value = snode.pop(int(sk))
    else: raise KeyError(f"Cannot move from {type(snode)}")
    insert(m, dst, value)

In [ ]:
tr2 = deepcopy(tr)

# Normal: move a leaf to another position
move(tr2, ['Apollo astronauts', 0], ['Apollo 11', 0])
test_eq(val_at(tr2, 'Apollo 11.0'), 'Neil Armstrong')

# Normal: move a subtree to another branch
move(tr2, ['Apollo astronauts', 1, 'other'], ['Apollo 11', 'relocated'])
test_eq(val_at(tr2, 'Apollo 11.6.relocated.0'), 'Bruce Wayne')

# Edge: move to existing dst → raises
test_fail(lambda: move(deepcopy(tr), ['Apollo astronauts'], ['Apollo 11']))

# Edge: move from non-existent src → raises
test_fail(lambda: move(tr2, ['ghost', 0], ['Apollo 11', 0]))

## more ops

Other common tree function/transformations to consider:

- **`map_leaves`** — apply a function to all leaves, returning a new tree (simpler API than `transform`)
- **`map_nodes`** — apply a function to all subtree nodes
- **`flatten`** — collapse the tree to a flat list of leaves (already partially covered by `walk`)
- **`merge`** — merge two forests, combining children of same-named roots
- **`depth`** / **`height`** — max depth of the tree
- **`count_leaves`** — count total leaves
- **`paths`** — yield just the paths (without values)
- **`from_paths`** — reconstruct a tree from `(path, leaf)` pairs (inverse of `walk`)
- **`zip_trees`** — zip two trees of same shape, yielding paired leaves
- **`reduce`** — fold over leaves with an accumulator

The most interesting and useful are probably `merge`, `from_paths` (inverse of `walk`), and `reduce`.

In [ ]:
def map_leaves(m: Tree, f: Callable) -> Tree:
    "Return new tree with f applied to every leaf"
    return transform(m, on_leaf=lambda p, v: f(v))

In [ ]:
def map_nodes(m: Tree, f: Callable) -> Tree:
    "Return new tree with f applied to every subtree node"
    return transform(m, on_node=lambda p, v: f(v))

In [ ]:
#| export
def reduce_leaves(m: Tree, f: Callable, init=None):
    "Reduce all leaves left-to-right with f(acc, val)"
    acc = init
    for _, v in walk(m): acc = f(acc, v)
    return acc

In [ ]:
test_eq(map_leaves(tr, lambda v: v.upper() if isinstance(v, str) else v),
        {'Apollo astronauts': ['NEIL ARMSTRONG', 'ALAN BEAN', {'other': ['BRUCE WAYNE', 'CLARK KENT', 'PETER PARKER']}, 'EDGAR MITCHELL', 'ALAN SHEPARD'],
         'Apollo 11': ['NEIL ARMSTRONG', 'BUZZ ALDRIN', 'EDGAR MITCHELL', {'a': 1, 'b': 2}, 'ALAN SHEPARD']})

test_eq(reduce_leaves(tr, lambda acc, v: acc + [v], []),
        [v for _, v in walk(tr)])

test_eq(reduce_leaves(tr, lambda acc, v: acc + (1 if isinstance(v, str) else 0), 0), 11)

test_eq(map_nodes(tr, lambda v: {k: vs for k, vs in v.items() if k != 'b'}),
        {'Apollo astronauts': ['Neil Armstrong', 'Alan Bean', {'other': ['Bruce Wayne', 'Clark Kent', 'Peter Parker']}, 'Edgar Mitchell', 'Alan Shepard'],
         'Apollo 11': ['Neil Armstrong', 'Buzz Aldrin', 'Edgar Mitchell', {'a': 1}, 'Alan Shepard']})

## example: file walker
A forest with key Paths

In [ ]:
#| export
exclude = ['.git', '__pycache__', '.ipynb_checkpoints']
_gett = attrgetter('name')

In [ ]:
@patch
def tree(self:Path, file_type=None, file_exts=None, exclude=None, hidden=False, max_level=None, on_file=None, on_dir=None) -> Tree:
    "Return a forest rooted at path with optional inclusions/exclusions and leaf/node transforms"
    import mimetypes
    extns=L(file_exts)
    if file_type: extns += L(k for k,v in mimetypes.types_map.items() if v.startswith(file_type+'/'))
    has_extns = len(extns)==0
    exclude = set(exclude or [])
    def _kids(p, lvl=0):
        if max_level is not None and lvl >= max_level: return []
        children = []
        for o in sorted(p.iterdir(), key=_gett):
            if o.name in exclude or (not hidden and o.name.startswith('.')): continue
            if o.is_dir(): children.append({o: _kids(o, lvl+1)} if on_dir is None else on_dir(o, _kids(o, lvl+1)))
            elif has_extns or o.suffix in extns: children.append(on_file(o) if on_file else o)
        return children
    return {self: _kids(self)}

In [ ]:
(Path.home()/'prj/pote').resolve().tree(file_exts=['.py'])

{Path('/app/data/prj/pote'): [{Path('/app/data/prj/pote/explorer'): [{Path('/app/data/prj/pote/explorer/importdlg'): [{Path('/app/data/prj/pote/explorer/importdlg/test_dialogs'): []}]},
    {Path('/app/data/prj/pote/explorer/isolated'): []},
    {Path('/app/data/prj/pote/explorer/js'): []},
    {Path('/app/data/prj/pote/explorer/super_w'): [{Path('/app/data/prj/pote/explorer/super_w/__pycache__'): []},
      Path('/app/data/prj/pote/explorer/super_w/super_w.py')]}]},
  {Path('/app/data/prj/pote/nbs'): [{Path('/app/data/prj/pote/nbs/data'): []},
    {Path('/app/data/prj/pote/nbs/static'): []}]},
  {Path('/app/data/prj/pote/pote'): [Path('/app/data/prj/pote/pote/__init__.py'),
    {Path('/app/data/prj/pote/pote/__pycache__'): []},
    Path('/app/data/prj/pote/pote/_modidx.py'),
    Path('/app/data/prj/pote/pote/all.py'),
    Path('/app/data/prj/pote/pote/basic.py'),
    Path('/app/data/prj/pote/pote/callback.py'),
    Path('/app/data/prj/pote/pote/common.py'),
    Path('/app/data/prj/pot

In [ ]:
(Path.home()/'prj/pote').resolve().tree(file_exts=['.py'], max_level=1)

{Path('/app/data/prj/pote'): [{Path('/app/data/prj/pote/explorer'): []},
  {Path('/app/data/prj/pote/nbs'): []},
  {Path('/app/data/prj/pote/pote'): []},
  {Path('/app/data/prj/pote/pote.egg-info'): []},
  {Path('/app/data/prj/pote/tool'): []}]}

In [ ]:
#| export
@patch
def tree(self:Path, 
    file_type:str=None,     # mime type to include, e.g. 'image'
    file_exts:list=None,    # file extensions to include, e.g. ['.py']
    exclude:list=None,      # names to exclude
    hidden:bool=False,      # include hidden files/dirs
    max_level:int=None,     # max depth to recurse; dirs at limit become unexpanded leaves
    on_file:Callable=None,  # replaces file leaf with return value; called with (path)
    on_dir:Callable=None,   # replaces dir node with return value; called with (path, children) or (path, None) for unexpanded dirs
) -> Tree:
    "Return a forest rooted at path with optional inclusions/exclusions and leaf/node transforms"
    import mimetypes
    extns=L(file_exts)
    if file_type: extns += L(k for k,v in mimetypes.types_map.items() if v.startswith(file_type+'/'))
    has_extns = len(extns)==0
    exclude = set(exclude or [])
    def _kids(p, lvl=0):
        if max_level is not None and lvl >= max_level: return []
        children = []
        for o in sorted(p.iterdir(), key=_gett):
            if o.name in exclude or (not hidden and o.name.startswith('.')): continue
            if o.is_dir():
                if max_level is not None and lvl+1 >= max_level: children.append(o if on_dir is None else on_dir(o, None))
                else: children.append({o: _kids(o, lvl+1)} if on_dir is None else on_dir(o, _kids(o, lvl+1)))
            elif has_extns or o.suffix in extns: children.append(on_file(o) if on_file else o)
        return children
    return {self: _kids(self)} if on_dir is None else on_dir(self, _kids(self))

In [ ]:
(Path.home()/'prj/pote').resolve().tree(file_exts=['.py'], max_level=1)

{Path('/app/data/prj/pote'): [Path('/app/data/prj/pote/explorer'),
  Path('/app/data/prj/pote/nbs'),
  Path('/app/data/prj/pote/pote'),
  Path('/app/data/prj/pote/pote.egg-info'),
  Path('/app/data/prj/pote/tool')]}

In [ ]:
tr = Path('..').tree(exclude=exclude)
nbs = tr[Path('..')][4]
nbs

{Path('../nbs'): [Path('../nbs/00_basic.ipynb'),
  Path('../nbs/00_dutil.ipynb'),
  Path('../nbs/00_project.ipynb'),
  Path('../nbs/01_display.ipynb'),
  Path('../nbs/01_flakes.ipynb'),
  Path('../nbs/01_tree.ipynb'),
  Path('../nbs/02_git.ipynb'),
  Path('../nbs/02_isolated.ipynb'),
  Path('../nbs/02_logger.ipynb'),
  Path('../nbs/02_logger_loguru.ipynb'),
  Path('../nbs/02_server.ipynb'),
  Path('../nbs/05_test.ipynb'),
  Path('../nbs/10_callback.ipynb'),
  Path('../nbs/15_config.ipynb'),
  Path('../nbs/17_display.ipynb'),
  Path('../nbs/20_widgets.ipynb'),
  Path('../nbs/_quarto.yml'),
  Path('../nbs/apilist.txt'),
  {Path('../nbs/data'): [Path('../nbs/data/log_across.ipynb'),
    Path('../nbs/data/test.ipynb')]},
  Path('../nbs/index.ipynb'),
  Path('../nbs/llms-ctx-full.txt'),
  Path('../nbs/llms-ctx.txt'),
  Path('../nbs/llms.txt'),
  Path('../nbs/nbdev.yml'),
  {Path('../nbs/static'): [Path('../nbs/static/apollo_astronauts.json'),
    Path('../nbs/static/file.txt'),
    Path('..

In [ ]:
# def dir_tree(path, exclude=None, on_file=None, on_dir=None) -> Tree:
#     "Return a forest rooted at path with optional exclusions and leaf/node transforms"
#     path = Path(path)
#     exclude = set(exclude or [])
#     def _kids(p):
#         children = []
#         for o in sorted(p.iterdir(), key=lambda x: x.name.lower()):
#             if o.name in exclude: continue
#             if o.is_dir(): children.append({o: _kids(o)} if on_dir is None else on_dir(o, _kids(o)))
#             else: children.append(on_file(o) if on_file else o)
#         return children
#     return {path: _kids(path)}

In [ ]:
# tr = dir_tree('..', exclude=exclude)
# nbs = tr[Path('..')][5]
# nbs

In [ ]:
prune(nbs, lambda v: v.suffix == '.ipynb')

{Path('../nbs'): [Path('../nbs/00_basic.ipynb'),
  Path('../nbs/00_dutil.ipynb'),
  Path('../nbs/00_project.ipynb'),
  Path('../nbs/01_display.ipynb'),
  Path('../nbs/01_flakes.ipynb'),
  Path('../nbs/01_tree.ipynb'),
  Path('../nbs/02_git.ipynb'),
  Path('../nbs/02_isolated.ipynb'),
  Path('../nbs/02_logger.ipynb'),
  Path('../nbs/02_logger_loguru.ipynb'),
  Path('../nbs/02_server.ipynb'),
  Path('../nbs/05_test.ipynb'),
  Path('../nbs/10_callback.ipynb'),
  Path('../nbs/15_config.ipynb'),
  Path('../nbs/17_display.ipynb'),
  Path('../nbs/20_widgets.ipynb'),
  {Path('../nbs/data'): [Path('../nbs/data/log_across.ipynb'),
    Path('../nbs/data/test.ipynb')]},
  Path('../nbs/index.ipynb')]}

In [ ]:
# tr = dir_tree('..', exclude=exclude, on_file=lambda o: o.name, on_dir=lambda d,c: {d.name: c})
# tr[Path('..')][5]

In [ ]:
Path('.').resolve().tree(on_file=lambda o: o.name, on_dir=lambda d,c: {d.name: c})

{'nbs': ['00_basic.ipynb',
  '00_dutil.ipynb',
  '00_project.ipynb',
  '01_display.ipynb',
  '01_flakes.ipynb',
  '01_tree.ipynb',
  '02_git.ipynb',
  '02_isolated.ipynb',
  '02_logger.ipynb',
  '02_logger_loguru.ipynb',
  '02_server.ipynb',
  '05_test.ipynb',
  '10_callback.ipynb',
  '15_config.ipynb',
  '17_display.ipynb',
  '20_widgets.ipynb',
  '_quarto.yml',
  'apilist.txt',
  {'data': ['log_across.ipynb', 'test.ipynb']},
  'index.ipynb',
  'llms-ctx-full.txt',
  'llms-ctx.txt',
  'llms.txt',
  'nbdev.yml',
  {'static': ['apollo_astronauts.json', 'file.txt', 'wordlist.txt']},
  'styles.css']}

In [ ]:
# def show_path(path, exclude=None):
#     t = dir_tree(path, exclude, on_file=lambda o: Li(o.name), on_dir=lambda o,ch: Li(Details(Summary(o.name), Ul(*ch))))
#     show(Ul(*next(iter(t.values()))))

In [ ]:
# show_path('..', exclude=exclude)

In [ ]:
# def print_path(path, exclude=None, color=True):
#     _d,_r,_g = ('\033[34m','\033[0m','\033[38;5;7m') if color else ('','','')
#     def _print(items, prefix=''):
#         items = list(items)
#         for i,o in enumerate(items):
#             last = i == len(items)-1
#             conn,ext = ('└─ ','   ') if last else ('├─ ','│  ')
#             if not isinstance(o, Mapping): print(_g + prefix + conn + _r + (o.name if isinstance(o, Path) else o))
#             else:
#                 k,ch = next(iter(o.items()))
#                 print(_g + prefix + conn + _r + _d + (k.name if isinstance(k, Path) else k) + _r)
#                 _print(ch, prefix+ext)
#     _print(next(iter(dir_tree(path, exclude).values())))

In [ ]:
# print_path('.', exclude=exclude)

Refactor `pprint` to generate any of the two outputs (indent, connectors), or even others, using transform and callbacks?

Good point — the idea is to make `pprint` style-agnostic by parameterizing the formatting through callbacks passed to `transform`. The indent style works naturally with `(path, value)`. The connector style is trickier though — it needs to know whether an item is the *last sibling*, which `transform` doesn't currently provide.

The key insight: `transform` callbacks gain a `ctx` dict (3rd arg) with `is_last`, `idx`, `n_sibs`, `depth`. Then `pprint` delegates to `transform` for both styles, using `ctx` to drive formatting.

Updated `transform`:

In [ ]:
#| export
def transform(tr, on_leaf=None, on_node_pre=None, on_node=None, p=()):
    "Walk forest applying callbacks with context dict, returning new tree"
    res = {}
    for k, vs in tr.items():
        children, cp, wrapped = [], p + (k,), not is_listy(vs)
        vs = vs if not wrapped else [vs]
        n_ch = len(vs)
        for i, v in enumerate(vs):
            ip, ctx = cp + (i,), dict(idx=i, n_sibs=n_ch, is_last=i==n_ch-1, depth=len(cp))
            if is_tree(v):
                if on_node_pre: on_node_pre(ip, v, ctx)
                sv = transform(v, on_leaf, on_node_pre, on_node, ip)
                r = on_node(ip, sv, ctx) if on_node else None
                children.append(r if r is not None else sv)
            else:
                r = on_leaf(ip, v, ctx) if on_leaf else None
                children.append(r if r is not None else v)
        res[k] = children[0] if wrapped else children
    if on_node and res and not p:
        r = on_node(p, res, dict(depth=len(p), idx=0, n_sibs=1, is_last=True))
        if r is not None: res = r
    return res

In [ ]:
nbsd = transform(nbs, on_leaf=lambda p,v,ctx: v.name, on_node=lambda p,v,ctx: {k.name:v for k,v in v.items()})
nbsd

{'nbs': ['00_basic.ipynb',
  '00_dutil.ipynb',
  '00_project.ipynb',
  '01_display.ipynb',
  '01_flakes.ipynb',
  '01_tree.ipynb',
  '02_git.ipynb',
  '02_isolated.ipynb',
  '02_logger.ipynb',
  '02_logger_loguru.ipynb',
  '02_server.ipynb',
  '05_test.ipynb',
  '10_callback.ipynb',
  '15_config.ipynb',
  '17_display.ipynb',
  '20_widgets.ipynb',
  '_quarto.yml',
  'apilist.txt',
  {'data': ['log_across.ipynb', 'test.ipynb']},
  'index.ipynb',
  'llms-ctx-full.txt',
  'llms-ctx.txt',
  'llms.txt',
  'nbdev.yml',
  {'static': ['apollo_astronauts.json', 'file.txt', 'wordlist.txt']},
  'styles.css']}

Updated helpers (callbacks now take 3 args):

In [ ]:
#| export
def map_leaves(tr, f): return transform(tr, on_leaf=lambda p, v, ctx: f(v))
def map_nodes(tr, f): return transform(tr, on_node=lambda p, v, ctx: f(v))

Updated `pprint` — with two styles delegate to `transform`:

In [ ]:
#| export
def _print_indent(tr, indent):
    _pre = lambda p, v, ctx: print(' '*indent*ctx['depth'] + str(next(iter(v))) + ':')
    _leaf = lambda p, v, ctx: print(' '*indent*ctx['depth'] + str(v))
    for k in tr: print(f"{k}:"); transform({k: tr[k]}, on_leaf=_leaf, on_node_pre=_pre)

def _print_tree(tr, color):
    _d,_r,_g = ('\033[34m','\033[0m','\033[38;5;7m') if color else ('','','')
    pres = []
    def _conn(ctx): return _g + ''.join(pres) + ('└─ ' if ctx['is_last'] else '├─ ') + _r
    def _leaf(p, v, ctx): print(_conn(ctx) + str(v))
    def _pre(p, v, ctx):
        print(_conn(ctx) + _d + str(next(iter(v))) + _r)
        pres.append('   ' if ctx['is_last'] else '│  ')
    def _post(p, v, ctx):
        if pres: pres.pop();
        return v
    for k, vs in tr.items():
        print(_d + str(k) + _r)
        pres.clear()
        transform({k: vs}, on_leaf=_leaf, on_node_pre=_pre, on_node=_post)
    
def pprint(tr, style='indent', indent=2, color=True):
    "Pretty print a forest in 'indent' or 'tree' style"
    if style == 'tree': _print_tree(tr, color)
    else: _print_indent(tr, indent)

In [ ]:
pprint(nbsd)

nbs:
  00_basic.ipynb
  00_dutil.ipynb
  00_project.ipynb
  01_display.ipynb
  01_flakes.ipynb
  01_tree.ipynb
  02_git.ipynb
  02_isolated.ipynb
  02_logger.ipynb
  02_logger_loguru.ipynb
  02_server.ipynb
  05_test.ipynb
  10_callback.ipynb
  15_config.ipynb
  17_display.ipynb
  20_widgets.ipynb
  _quarto.yml
  apilist.txt
  data:
      log_across.ipynb
      test.ipynb
  index.ipynb
  llms-ctx-full.txt
  llms-ctx.txt
  llms.txt
  nbdev.yml
  static:
      apollo_astronauts.json
      file.txt
      wordlist.txt
  styles.css


In [ ]:
pprint(nbsd, style='tree')

nbs
├─ 00_basic.ipynb
├─ 00_dutil.ipynb
├─ 00_project.ipynb
├─ 01_display.ipynb
├─ 01_flakes.ipynb
├─ 01_tree.ipynb
├─ 02_git.ipynb
├─ 02_isolated.ipynb
├─ 02_logger.ipynb
├─ 02_logger_loguru.ipynb
├─ 02_server.ipynb
├─ 05_test.ipynb
├─ 10_callback.ipynb
├─ 15_config.ipynb
├─ 17_display.ipynb
├─ 20_widgets.ipynb
├─ _quarto.yml
├─ apilist.txt
├─ data
│  ├─ log_across.ipynb
│  └─ test.ipynb
├─ index.ipynb
├─ llms-ctx-full.txt
├─ llms-ctx.txt
├─ llms.txt
├─ nbdev.yml
├─ static
│  ├─ apollo_astronauts.json
│  ├─ file.txt
│  └─ wordlist.txt
└─ styles.css


The tree-style connector logic works via a `pres` prefix stack — `on_node_pre` pushes, `on_node` (post) pops. Root keys are handled in the outer loop since they sit above `transform`'s iteration.

Both print styles simplify to thin wrappers around `dir_tree` + `pprint(..., style='tree')`.

In [ ]:
#| export
def _details_transform():
    def _leaf(p, v, ctx): return Li(v.name)
    def _node(p, v, ctx):
        wrp, nm = (Li, next(iter(v)).name) if p else (noop, next(iter(v)).resolve().name)
        return wrp(Details(Summary(nm), Ul(*next(iter(v.values()))), open=not bool(p)))
    return dict(on_leaf=_leaf, on_node=_node)

def show_path(path, exclude=None):
    tr = Path(path).tree(exclude=exclude)
    show(transform(tr, **_details_transform()))

In [ ]:
show_path('.', exclude=exclude)

HTML(<details open><summary>nbs</summary>  <ul>
    <li>00_basic.ipynb</li>
    <li>00_dutil.ipynb</li>
    <li>00_project.ipynb</li>
    <li>01_display.ipynb</li>
    <li>01_flakes.ipynb</li>
    <li>01_tree.ipynb</li>
    <li>02_git.ipynb</li>
    <li>02_isolated.ipynb</li>
    <li>02_logger.ipynb</li>
    <li>02_logger_loguru.ipynb</li>
    <li>02_server.ipynb</li>
    <li>05_test.ipynb</li>
    <li>10_callback.ipynb</li>
    <li>15_config.ipynb</li>
    <li>17_display.ipynb</li>
    <li>20_widgets.ipynb</li>
    <li>_quarto.yml</li>
    <li>apilist.txt</li>
    <li>
<details><summary>data</summary>        <ul>
          <li>log_across.ipynb</li>
          <li>test.ipynb</li>
        </ul>
</details>    </li>
    <li>index.ipynb</li>
    <li>llms-ctx-full.txt</li>
    <li>llms-ctx.txt</li>
    <li>llms.txt</li>
    <li>nbdev.yml</li>
    <li>
<details><summary>static</summary>        <ul>
          <li>apollo_astronauts.json</li>
          <li>file.txt</li>
          <li>wordlist.txt</li>
        </ul>
</details>    </li>
    <li>styles.css</li>
  </ul>
</details>)

In [ ]:
show(transform(Path('.').tree(exclude=exclude), **_details_transform()))

HTML(<details open><summary>nbs</summary>  <ul>
    <li>00_basic.ipynb</li>
    <li>00_dutil.ipynb</li>
    <li>00_project.ipynb</li>
    <li>01_display.ipynb</li>
    <li>01_flakes.ipynb</li>
    <li>01_tree.ipynb</li>
    <li>02_git.ipynb</li>
    <li>02_isolated.ipynb</li>
    <li>02_logger.ipynb</li>
    <li>02_logger_loguru.ipynb</li>
    <li>02_server.ipynb</li>
    <li>05_test.ipynb</li>
    <li>10_callback.ipynb</li>
    <li>15_config.ipynb</li>
    <li>17_display.ipynb</li>
    <li>20_widgets.ipynb</li>
    <li>_quarto.yml</li>
    <li>apilist.txt</li>
    <li>
<details><summary>data</summary>        <ul>
          <li>log_across.ipynb</li>
          <li>test.ipynb</li>
        </ul>
</details>    </li>
    <li>index.ipynb</li>
    <li>llms-ctx-full.txt</li>
    <li>llms-ctx.txt</li>
    <li>llms.txt</li>
    <li>nbdev.yml</li>
    <li>
<details><summary>static</summary>        <ul>
          <li>apollo_astronauts.json</li>
          <li>file.txt</li>
          <li>wordlist.txt</li>
        </ul>
</details>    </li>
    <li>styles.css</li>
  </ul>
</details>)

Alright, finally, is there any library out there that does something similar to what we've created here? For example, for `show_tree` we could simply use `tree` command (though much less versatil).

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, a few relevant ones:

- **`tree` command** — as you noted, the simplest option (`!tree -I '__pycache__'`), but no Python integration or customization.

- **[Rich](https://rich.readthedocs.io/en/latest/tree.html)** — probably the closest to our `print_tree`. [*](https://rich.readthedocs.io/en/latest/tree.html "Tree — Rich 14.1.0 documentation") Rich has a `Tree` class that generates a tree view in the terminal, great for presenting filesystem contents or any hierarchical data, where each branch can have a label that's text or any other Rich renderable. It's very polished with colors and styles, but you build the tree manually node by node rather than from a path.

- **[treelib](https://hackersandslackers.com/python-tree-hierachies-treelib/)** — [*](https://hackersandslackers.com/python-tree-hierachies-treelib/ "Visualize Folder Structures with Python’s Treelib") a Python library that allows you to create a visual tree hierarchy, a plaintext representation of parent-child relationships. More general-purpose (any tree, not just filesystem), but again requires manual construction.

- **[directory-tree](https://pypi.org/project/directory-tree/)** — [*](https://pypi.org/project/directory-tree/ "directory-tree · PyPI") a package specifically for printing out the tree structure of a user-defined directory. Simple, but no interactivity or HTML output.

None of them offer the **collapsible HTML output** of our `show_tree`, which is really the most useful feature in a notebook context — that part is genuinely novel.

<details class='token-usage-details'><summary>$0.2727</summary>

`total=94,925 | in=94,446 | out=479 | cached=51.3% | cache_new=29,567 | searches=1 | $0.2727`

</details>


## file metadata

###### start-section

In [ ]:
class FileNode(FC.GetAttr):
    "Node in a file tree"
    _default='path'
    def __init__(self, path, **kwargs):
        self.path = Path(path).resolve(); self._arg = path
        if kwargs: self._meta = kwargs
    def __str__(self): return str(self.path)
    def __repr__(self): return f'{self.__class__.__name__}({self._arg})'
    @property
    def meta(self):
        if not hasattr(self, '_meta'): self._meta = FC.AttrDict()
        return self._meta

In [ ]:
fnd = FileNode('.')
fnd.name, str(fnd), str(fnd.resolve()), type(fnd.resolve())

('nbs', '/app/data/prj/pote/nbs', '/app/data/prj/pote/nbs', pathlib.PosixPath)

In [ ]:
fnd.meta

```python
{}
```

In [ ]:
fnd.meta.open = True
fnd.meta

```python
{'open': True}
```

In [ ]:
tr = Path('.').tree(exclude=exclude, on_file=lambda o: FileNode(o))
tr

{Path('.'): [FileNode(00_basic.ipynb),
  FileNode(00_dutil.ipynb),
  FileNode(00_project.ipynb),
  FileNode(01_display.ipynb),
  FileNode(01_flakes.ipynb),
  FileNode(01_tree.ipynb),
  FileNode(02_git.ipynb),
  FileNode(02_isolated.ipynb),
  FileNode(02_logger.ipynb),
  FileNode(02_logger_loguru.ipynb),
  FileNode(02_server.ipynb),
  FileNode(05_test.ipynb),
  FileNode(10_callback.ipynb),
  FileNode(15_config.ipynb),
  FileNode(17_display.ipynb),
  FileNode(20_widgets.ipynb),
  FileNode(_quarto.yml),
  FileNode(apilist.txt),
  {Path('data'): [FileNode(data/log_across.ipynb), FileNode(data/test.ipynb)]},
  FileNode(index.ipynb),
  FileNode(llms-ctx-full.txt),
  FileNode(llms-ctx.txt),
  FileNode(llms.txt),
  FileNode(nbdev.yml),
  {Path('static'): [FileNode(static/apollo_astronauts.json),
    FileNode(static/file.txt),
    FileNode(static/wordlist.txt)]},
  FileNode(styles.css)]}

I want to assign metadata payload to file paths. `FileNode` is a possible solution, but has limitations. How would you implement this?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The main limitation of `FileNode` is that `isinstance(o, Path)` fails, so it breaks existing code that checks for `Path`. The cleanest fix is to subclass `Path` directly — Python 3.12 supports this without tricks:

###### end-section

In [ ]:
#| export
class FileMeta(Path):
    "Path with attached metadata dict"
    def __new__(cls, *args, **kwargs): return super().__new__(cls, *args)
    def __init__(self, *args, **kwargs):
        super().__init__(*args)
        self.meta = AttrDict(kwargs)

In [ ]:
f = FileMeta('00_basic.ipynb', open=True, tags=['nb'])
isinstance(f, Path), f.name, f.meta

(True, '00_basic.ipynb', {'open': True, 'tags': ['nb']})

In [ ]:
f.__class__.mro()

[__main__.FileMeta, pathlib.Path, pathlib.PurePath, object]

In [ ]:
d = FileMeta('.', open=True, tags=['nb'])
str(d.resolve()), type(d.resolve()), d.is_dir()

('/app/data/prj/pote/nbs', __main__.FileMeta, True)

In [ ]:
tr = Path('.').tree(exclude=exclude, on_file=lambda o: FileMeta(o), on_dir=lambda d,c: {FileMeta(d): c})
tr

{Path('.'): [Path('00_basic.ipynb'),
  Path('00_dutil.ipynb'),
  Path('00_project.ipynb'),
  Path('01_display.ipynb'),
  Path('01_flakes.ipynb'),
  Path('01_tree.ipynb'),
  Path('02_git.ipynb'),
  Path('02_isolated.ipynb'),
  Path('02_logger.ipynb'),
  Path('02_logger_loguru.ipynb'),
  Path('02_server.ipynb'),
  Path('05_test.ipynb'),
  Path('10_callback.ipynb'),
  Path('15_config.ipynb'),
  Path('17_display.ipynb'),
  Path('20_widgets.ipynb'),
  Path('_quarto.yml'),
  Path('apilist.txt'),
  {Path('data'): [Path('data/log_across.ipynb'), Path('data/test.ipynb')]},
  Path('index.ipynb'),
  Path('llms-ctx-full.txt'),
  Path('llms-ctx.txt'),
  Path('llms.txt'),
  Path('nbdev.yml'),
  {Path('static'): [Path('static/apollo_astronauts.json'),
    Path('static/file.txt'),
    Path('static/wordlist.txt')]},
  Path('styles.css')]}

`FileMeta` should override some of the methods that return paths to copy over the metadata if necessary. I think these ones:
- absolute, relative_to, resolve
- with_name, with_stem, with_suffix
- copy, copy_into, if `preserve_metadata` is True
- rename
- move, move_into
- replace??, not sure (in fact I don't know what replace does)

But let's discuss.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few thoughts to guide the discussion:

**Always copy metadata (same file, different path representation):**
- `resolve`, `absolute`, `relative_to` — these just change how the same file is addressed, so copying metadata unconditionally makes sense

**Probably don't copy (different file entirely):**
- `with_name`, `with_stem`, `with_suffix` — the result points to a *different* file that may not exist yet or may have nothing to do with the original. Copying metadata silently could be surprising. I'd leave these alone, or only copy if the user explicitly subclasses further.

**`preserve_metadata` flag makes sense:**
- `copy`, `copy_into` — you're duplicating the file, so optionally carrying metadata over is reasonable
- `rename`, `move`, `move_into` — the file is the same content at a new location, so you probably *do* want metadata preserved by default here, unlike copy

**`replace`:** it's like `rename` but overwrites the target atomically if it exists. So it's a destructive move — the *source* file's identity wins. I'd treat it like `rename` and preserve metadata by default.

One broader question worth settling first: should `FileMeta` metadata be considered **file-identity metadata** (travels with the file wherever it goes) or **context metadata** (only meaningful in the specific place it was created)? That determines how aggressive you want to be with propagation.

<details class='token-usage-details'><summary>$0.1513</summary>

`total=25,470 | in=24,687 | out=783 | cached=0.0% | cache_new=20,125 | $0.1513`

</details>


It is context metadata. For instance, a tree visor component can store open/close state or selection state, or allow in-place renaming. The metadata copy-over is a good-to-have helper for that kind of uses, but we can perfectly let the consumer app track the app state itself. So, helpers, not file state.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That clarifies things nicely. So `FileMeta` is really about **UI/view state** attached to a path for convenience, not persistent file identity. That simplifies the design considerably:

- **`resolve`, `absolute`, `relative_to`** — worth overriding to preserve metadata, since these are the most common "same file, different representation" operations that a tree viewer would naturally call
- **`with_name`, `with_stem`, `with_suffix`, `replace`** — skip, they imply a different file/identity
- **`rename`, `move`, `move_into`, `copy`, `copy_into`** — skip too; if the app is tracking state itself, it can re-attach metadata after these operations as needed. No need for a `preserve_metadata` flag.

So really just the three path-representation methods are worth overriding. Shall we go ahead with those?

<details class='token-usage-details'><summary>$0.1457</summary>

`total=25,348 | in=25,144 | out=204 | cached=0.0% | cache_new=24,892 | $0.1457`

</details>


In [ ]:
#| export
@patch
def resolve(self:FileMeta, strict=False): return FileMeta(super().resolve(strict=strict), **self.meta)

@patch
def absolute(self:FileMeta): return FileMeta(super().absolute(), **self.meta)

@patch
def relative_to(self:FileMeta, *other): return FileMeta(super().relative_to(*other), **self.meta)

# export -

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dutil import dlg_export
# dlg_export()